In [2]:
import numpy as np

In [3]:
class Node:

    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf(self):
        return self.value is not None


In [4]:
class DecisionTree:

    def __init__(self, max_depth=10, min_samples_split=2, criterion="gini"):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.criterion = criterion
        self.root = None
    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        self.root = self._grow_tree(X, y, 0)
        return self
    def predict(self, X):
        X = np.asarray(X)
        return np.array([self._traverse_tree(x, self.root) for x in X])

In [5]:
def _grow_tree(self, X, y, depth):

    n_samples, n_features = X.shape

    if (depth >= self.max_depth or n_samples < self.min_samples_split or len(np.unique(y)) == 1):
        return Node(value=self._majority_class(y))

    feature, threshold, gain = self._best_split(X, y)

    if gain <= 0:
        return Node(value=self._majority_class(y))

    left_mask = X[:, feature] <= threshold
    right_mask = X[:, feature] > threshold

    left = self._grow_tree(X[left_mask], y[left_mask], depth + 1)

    right = self._grow_tree(
        X[right_mask],
        y[right_mask],
        depth + 1
    )
    return Node(
        feature=feature,
        threshold=threshold,
        left=left,
        right=right
    )

DecisionTree._grow_tree = _grow_tree

In [6]:
def _gini(self, y):
    if len(y) == 0:
        return 0
    probs = np.bincount(y) / len(y)
    return 1 - np.sum(probs ** 2)


DecisionTree._gini = _gini

In [7]:
def _impurity(self, y):
    return self._gini(y)


DecisionTree._impurity = _impurity

In [8]:
def _information_gain(self, y, values, threshold):

    parent = self._impurity(y)

    left_mask = values <= threshold
    right_mask = values > threshold

    if left_mask.sum() == 0 or right_mask.sum() == 0:
        return 0

    y_left = y[left_mask]
    y_right = y[right_mask]

    left_weight = len(y_left) / len(y)
    right_weight = len(y_right) / len(y)
    child = (
        left_weight * self._impurity(y_left)
        + right_weight * self._impurity(y_right)
    )

    return parent - child

DecisionTree._information_gain = _information_gain

In [9]:
def _traverse_tree(self, x, node):
    if node.is_leaf():
        return node.value
    if x[node.feature] <= node.threshold:
        return self._traverse_tree(x, node.left)
    return self._traverse_tree(x, node.right)

DecisionTree._traverse_tree = _traverse_tree

In [10]:
def _best_split(self, X, y):

    n_samples, n_features = X.shape

    best_gain = -1
    best_feature = None
    best_threshold = None

    for feature in range(n_features):

        values = X[:, feature]

        for threshold in np.unique(values):

            gain = self._information_gain(
                y,
                values,
                threshold
            )

            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold, best_gain


DecisionTree._best_split = _best_split

In [11]:
def _majority_class(self, y):
    return np.bincount(y).argmax()

DecisionTree._majority_class = _majority_class

In [1]:
class RandomForest:
    def __init__(self, n_estimators=10, max_depth=10, min_samples_split=2, max_features=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.trees = []

    def predict(self, X):
        X = np.asarray(X)
        # Gather predictions from all tree
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
    
        tree_preds = tree_preds.T
        majority_preds = np.array([np.bincount(sample_pred).argmax() for sample_pred in tree_preds])
        
        return majority_preds

In [2]:
def fit(self, X, y):
    X = np.asarray(X)
    y = np.asarray(y)
    n_samples, n_features = X.shape
    self.trees = []

    if self.max_features is None:
        self.max_features = int(np.sqrt(n_features))
    elif isinstance(self.max_features, float):
        self.max_features = int(self.max_features * n_features)

    for _ in range(self.n_estimators):
        boot_indices = np.random.choice(n_samples, size=n_samples, replace=True)
        X_boot = X[boot_indices]
        y_boot = y[boot_indices]
        tree = DecisionTree(
            max_depth=self.max_depth, 
            min_samples_split=self.min_samples_split
        )
    
        def _patched_best_split(X_sub, y_sub):
            n_samp, n_feat = X_sub.shape
            best_gain = -1
            best_feature = None
            best_threshold = None
            
            features_idx = np.random.choice(n_feat, size=min(self.max_features, n_feat), replace=False)

            for feature in features_idx:
                values = X_sub[:, feature]
                for threshold in np.unique(values):
                    gain = tree._information_gain(y_sub, values, threshold)
                    if gain > best_gain:
                        best_gain = gain
                        best_feature = feature
                        best_threshold = threshold

            return best_feature, best_threshold, best_gain

        tree._best_split = _patched_best_split
        
        tree.fit(X_boot, y_boot)
        self.trees.append(tree)
        
    return self